In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import pickle
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from PIL import Image
import matplotlib.pyplot as plt

from tqdm import tqdm

In [3]:
BASE = "/content/drive/MyDrive/MultiModalProject/Flickr8k"

TRAIN_CSV = os.path.join(BASE, "dataset_caption_train.csv")
VAL_CSV   = os.path.join(BASE, "dataset_caption_val.csv")
TEST_CSV  = os.path.join(BASE, "dataset_caption_test.csv")

TRAIN_FEATURES = os.path.join(BASE, "efficient_train.npy")
VAL_FEATURES   = os.path.join(BASE, "efficient_val.npy")
TEST_FEATURES  = os.path.join(BASE, "efficient_test.npy")

MODEL_DIR = os.path.join(BASE, "models", "GRU")

os.makedirs(MODEL_DIR, exist_ok=True)

In [4]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(29112, 2)
(3238, 2)
(8095, 2)


In [5]:
train_features = np.load(
    TRAIN_FEATURES,
    allow_pickle=True
).item()

val_features = np.load(
    VAL_FEATURES,
    allow_pickle=True
).item()

test_features = np.load(
    TEST_FEATURES,
    allow_pickle=True
).item()

print(len(train_features))
print(len(val_features))
print(len(test_features))

5824
648
1619


In [6]:
counter = Counter()

for cap in train_df["caption"]:

    words = cap.split()

    counter.update(words)

vocab = {

    "<pad>":0,
    "<start>":1,
    "<end>":2,
    "<unk>":3

}

index = 4

for word, count in counter.items():

    if word in ["<pad>", "<start>", "<end>", "<unk>"]:
        continue

    if count >= 2:

        vocab[word] = index

        index += 1

idx2word = {
    v:k
    for k,v in vocab.items()
}

print("Vocab Size:", len(vocab))
print("Last Index:", max(vocab.values()))

Vocab Size: 4454
Last Index: 4453


In [7]:
with open(
    os.path.join(MODEL_DIR, "vocab.pkl"),
    "wb"
) as f:

    pickle.dump(vocab, f)

In [8]:
def encode_caption(caption):

    words = caption.split()

    encoded = []

    for word in words:

        encoded.append(

            vocab.get(
                word,
                vocab["<unk>"]
            )

        )

    return torch.tensor(encoded)

In [9]:
train_df["encoded"] = train_df["caption"].apply(
    encode_caption
)

val_df["encoded"] = val_df["caption"].apply(
    encode_caption
)

test_df["encoded"] = test_df["caption"].apply(
    encode_caption
)

print(train_df.iloc[0]["caption"])
print(train_df.iloc[0]["encoded"])

<start> a child in a pink dress is climbing up a set of stairs in an entry way <end>
tensor([ 1,  4,  5,  6,  4,  7,  8,  9, 10, 11,  4, 12, 13, 14,  6, 15,  3, 16,
         2])


In [10]:
class CaptionDataset(Dataset):

    def __init__(
        self,
        dataframe,
        feature_dict
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.features = feature_dict

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = row["image"]

        feature = torch.tensor(

            self.features[image]

        ).float()

        caption = row["encoded"]

        return feature, caption

In [11]:
def collate_fn(batch):

    images = []

    captions = []

    for img, cap in batch:

        images.append(img)

        captions.append(cap)

    images = torch.stack(images)

    captions = pad_sequence(

        captions,

        batch_first=True,

        padding_value=vocab["<pad>"]

    )

    return images, captions


train_loader = DataLoader(

    CaptionDataset(
        train_df,
        train_features
    ),

    batch_size=32,

    shuffle=True,

    collate_fn=collate_fn

)

val_loader = DataLoader(

    CaptionDataset(
        val_df,
        val_features
    ),

    batch_size=32,

    shuffle=False,

    collate_fn=collate_fn

)

test_loader = DataLoader(

    CaptionDataset(
        test_df,
        test_features
    ),

    batch_size=32,

    shuffle=False,

    collate_fn=collate_fn

)

images, captions = next(
    iter(train_loader)
)

print(images.shape)
print(captions.shape)

torch.Size([32, 49, 1280])
torch.Size([32, 21])
